In [ ]:
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim
import torch
import os
import sys
import numpy as np

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_color_analysis import *
from syn_project.utils_notebook import *

# ---- helpers ----

def compute_reconstruction_quality(original_rgb, decoded_rgb):
    """
    original_rgb, decoded_rgb : tensors ou numpy (N, H, W, 3) ou (N, 3, H, W), valeurs dans [0,1].
    Retourne dict avec mse, ssim_mean.
    """
    # → numpy (N, H, W, 3) dans [0,1]
    def to_nhwc(x):
        if isinstance(x, torch.Tensor):
            x = x.detach().cpu().numpy()
        if x.ndim == 4 and x.shape[1] == 3:   # (N,3,H,W)
            x = x.transpose(0, 2, 3, 1)
        return x.astype(np.float32)

    orig = to_nhwc(original_rgb)
    dec  = to_nhwc(decoded_rgb)

    mse  = float(np.mean((orig - dec) ** 2))
    ssim_scores = [
        ssim(orig[i], dec[i], channel_axis=-1, data_range=1.0)
        for i in range(len(orig))
    ]
    return {"mse": mse, "ssim": float(np.mean(ssim_scores))}


# ---- boucle principale ----

conditions = [
    "ablation_cont",
    "ablation_cont_cy",
    "ablation_cont_cy_dcy",
    "ablation_cont_cy_dcy_trans",
    "ablation_cont_cy_trans",
    "ablation_cont_dcy",
    "ablation_cont_dcy_trans",
    "ablation_cont_trans",
    "ablation_cy",
    "ablation_cy_dcy",
    "ablation_cy_dcy_trans",
    "ablation_cy_trans",
    "ablation_dcy",
    "ablation_dcy_trans",
    "ablation_trans",
    "ablation_low_cont",
    "ablation_low_cy",
    "ablation_low_dcy",
    "ablation_low_trans",
]

checkpoint_epoch  = 0
n_samples_test    = 1000
split             = "test"
dataset           = "biased_00"

rows = []

for condition in conditions:
    print(f"\n=== {condition} ===")
    with total_silence():
        (global_workspace, domain_mods, gw_mod,
         visual_module, original_data,
         latent_domains, modules_name) = get_modules_data_from_exp(
            experiment_name=condition,
            n_samples_test=n_samples_test,
            split=split,
            checkpoint_epoch=checkpoint_epoch,
        )

    for s in [True, False]:
        objects = get_objects_from_v_latents(
            latent_domains, gw_mod, global_workspace, modules_name,
            modality_from='attr', modality_through='color',
            modality_main=['attr'], modality_add='color',
            start_v=s,
        )

        original_images_rgb = visual_module.decode_images(original_data['v_latents'])

        cat = []
        if 'attr' in modules_name:
            cat = original_data['attr'][0]
        if 'cat' in modules_name:
            cat = original_data['cat']

        # vision2 = la reconstruction finale qui t'intéresse
        decoded_images = visual_module.decode_images(objects['vision2'])

        # --- qualité de reconstruction ---
        recon = compute_reconstruction_quality(original_images_rgb, decoded_images)

        # --- analyse couleur / LDA ---
        colors_np = np.clip((objects['x2']['color'].detach().cpu().numpy() + 1) / 2, 0, 1) 
        cats      = cat.argmax(dim=1).detach().cpu().numpy()
        metrics, _ = hue_analysis(colors_np, cats,
                                cat_names=CAT_NAMES,
                                value=0.75, saturation_boost=1.8)
        results = logistic_probe(colors_np, cats)

        rows.append({
            "condition"         : condition,
            "start_v"           : s,
            "ssim"              : recon["ssim"],
            "lda_score"         : metrics["lda_score"],
        })

        del decoded_images, objects   # libère la mémoire GPU
        torch.cuda.empty_cache()

# ---- tableau récapitulatif ----

df = pd.DataFrame(rows).set_index("condition")
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)


=== ablation_cont ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_cont_cy ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_cont_cy_dcy ===


...


=== ablation_low_dcy ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_low_trans ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


KeyError: "['probe_accuracy'] not in index"

In [12]:
df = pd.DataFrame(rows).set_index(["condition", "start_v"] )
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)

In [10]:
df

,start_v,ssim,lda_score
condition,,,
ablation_cont,True,0.538120,0.332
ablation_cont,False,0.554566,0.454
ablation_cont_cy,True,0.795684,0.397
ablation_cont_cy,False,0.597435,0.946
ablation_cont_cy_dcy,True,0.860693,0.758
ablation_cont_cy_dcy,False,0.907903,0.756
ablation_cont_cy_dcy_trans,True,0.890742,0.975
ablation_cont_cy_dcy_trans,False,0.953491,0.989
ablation_cont_cy_trans,True,0.864799,0.838
